In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (27 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached https://download.pytorch.org/whl/sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached https://download.pytorch.org/whl/networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached https://download.pytorch.org/whl/fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/pillow-11.3.0-cp312-cp312-win_amd64.whl.meta

In [16]:
from app.vectorstore import init_vectorstore, store_text_vector, create_collection
from app.utils import embed_text, process_document

# paths = ["docs/EAU-EANM-ESTRO-ESUR-ISUP-SIOG-Guidelines-on-Prostate-Cancer-2025_updated.pdf",
#   "docs/EAU-Guidelines-on-Testicular-Cancer-2025.pdf",
#   "docs/EAU-Guidelines-on-Upper-Urinary-Tract-Urothelial-Carcinoma-2025_2025-06-02-054038_pezz.pdf"]

paths = ["docs/file1.pdf",
  "docs/file2.pdf",
  "docs/file3.pdf"]

client = init_vectorstore()
create_collection(client)
success = 0
for path in paths:
    try:
        for chunk in process_document(path=path):
          chunk_text_vector = embed_text(text=chunk["text"])
          store_text_vector(client,
                              collection_name="docs",
                              vector=chunk_text_vector,
                              text=chunk["text"],
                              payload={"doc": chunk["doc"], "page": chunk["page"], "text": chunk["text"]}
                              )
        success+=1
    except Exception as e:
        print(e)
        continue

Collection: docs, already exists


In [2]:
import torch
torch.cuda.is_available()

False

In [13]:
from app.vectorstore import query_top_k, init_vectorstore
from app.utils import embed_text

client = init_vectorstore()

query_text = "When can biopsy be omitted after a negative MRI?"
answers = query_top_k(client,"docs",embed_text(query_text))

In [17]:
print(answers[0].payload.get('text'))

effects, is questionable [404, 405]. 
The ‘MRI pathway’, in which patients with a positive MRI undergo only MRI-targeted biopsy and 
patients with a negative MRI are not biopsied at all, could avoid biopsy in 21-49% of the patients if a PI-RADS 
threshold of ≥ 3 is used to trigger biopsy [126, 208, 210, 314], at the cost of missing some significant cancers, 
especially in biopsy-naïve patients or in highly selected populations with high prevalence of csPCa (in which the 
MRI NPV decreases) [311, 406]. 
Adding perilesional sampling to targeted biopsy could mitigate the drawbacks of the ‘MRI pathway’ 
by maintaining good detection of csPCa while decreasing the over-diagnosis of insignificant cancer (see section 
5.5.4). Due to the low NPV of MRI in high risk populations, systematic biopsies are still necessary in patients 
with negative MRI and high clinical suspicion of PCa e.g., high PSA density.


In [15]:
for answer in answers:
    print(answer.payload.get('text'))

effects, is questionable [404, 405]. 
The ‘MRI pathway’, in which patients with a positive MRI undergo only MRI-targeted biopsy and 
patients with a negative MRI are not biopsied at all, could avoid biopsy in 21-49% of the patients if a PI-RADS 
threshold of ≥ 3 is used to trigger biopsy [126, 208, 210, 314], at the cost of missing some significant cancers, 
especially in biopsy-naïve patients or in highly selected populations with high prevalence of csPCa (in which the 
MRI NPV decreases) [311, 406]. 
Adding perilesional sampling to targeted biopsy could mitigate the drawbacks of the ‘MRI pathway’ 
by maintaining good detection of csPCa while decreasing the over-diagnosis of insignificant cancer (see section 
5.5.4). Due to the low NPV of MRI in high risk populations, systematic biopsies are still necessary in patients 
with negative MRI and high clinical suspicion of PCa e.g., high PSA density.
frequency of AS biopsy schedules: low PSA-D [575, 585-587], low PSA velocity (PSAV) [588, 

In [24]:
answers[0]

ScoredPoint(id='82e2bbcb-129f-5c83-a4f6-57ee4cb9d2d7', version=1896, score=0.65417325, payload={'doc': 'docs/EAU-EANM-ESTRO-ESUR-ISUP-SIOG-Guidelines-on-Prostate-Cancer-2025_updated.pdf', 'page': 45, 'text': 'effects, is questionable [404, 405]. \nThe ‘MRI pathway’, in which patients with a positive MRI undergo only MRI-targeted biopsy and \npatients with a negative MRI are not biopsied at all, could avoid biopsy in 21-49% of the patients if a PI-RADS \nthreshold of ≥ 3 is used to trigger biopsy [126, 208, 210, 314], at the cost of missing some significant cancers, \nespecially in biopsy-naïve patients or in highly selected populations with high prevalence of csPCa (in which the \nMRI NPV decreases) [311, 406]. \nAdding perilesional sampling to targeted biopsy could mitigate the drawbacks of the ‘MRI pathway’ \nby maintaining good detection of csPCa while decreasing the over-diagnosis of insignificant cancer (see section \n5.5.4). Due to the low NPV of MRI in high risk populations, sys

In [38]:
context = extract_context_from_answers(answers, threshold=0.62)

In [39]:
print(context)

effects, is questionable [404, 405]. 
The ‘MRI pathway’, in which patients with a positive MRI undergo only MRI-targeted biopsy and 
patients with a negative MRI are not biopsied at all, could avoid biopsy in 21-49% of the patients if a PI-RADS 
threshold of ≥ 3 is used to trigger biopsy [126, 208, 210, 314], at the cost of missing some significant cancers, 
especially in biopsy-naïve patients or in highly selected populations with high prevalence of csPCa (in which the 
MRI NPV decreases) [311, 406]. 
Adding perilesional sampling to targeted biopsy could mitigate the drawbacks of the ‘MRI pathway’ 
by maintaining good detection of csPCa while decreasing the over-diagnosis of insignificant cancer (see section 
5.5.4). Due to the low NPV of MRI in high risk populations, systematic biopsies are still necessary in patients 
with negative MRI and high clinical suspicion of PCa e.g., high PSA density.
